# 🧑‍⚖️ Human-in-the-Loop (HITL)

**HITL** stands for **Human-in-the-Loop** — a pattern where an AI agent **pauses before taking a consequential action** and waits for a human to review it, rather than acting fully autonomously.

## 🤔 Why do we need it?

Agents can call tools that have **real-world side effects**: sending emails, making payments, deleting records, posting to production. If the LLM gets it wrong, there's no undo. HITL puts a human checkpoint in front of exactly those risky steps, while letting **safe, read-only actions run automatically**.

## ⚙️ How it works in LangChain

`HumanInTheLoopMiddleware` lets you mark which tools require approval via `interrupt_on`:

| Tool | `interrupt_on` value | Behavior |
|---|---|---|
| `read_email` | `False` | Runs automatically (read-only, safe) |
| `send_email` | `True` | **Pauses** and waits for a human decision |

When a guarded tool is about to run, the agent **interrupts** — `invoke()` returns with an `__interrupt__` payload instead of finishing. Execution is saved by the `checkpointer` (here `InMemorySaver`), tied to a `thread_id`. The human then resumes with a `Command(resume=...)` on the **same thread**, choosing one of three decisions:

| Decision | Meaning |
|---|---|
| ✅ **approve** | Run the tool exactly as the agent proposed |
| ❌ **reject** | Cancel the tool call, optionally send feedback to the agent |
| ✏️ **edit** | Modify the tool's arguments, then run the edited version |

## 🔁 The flow at a glance

```
invoke() → LLM wants to call send_email
        → middleware interrupts  ⏸️
        → returns __interrupt__ (agent is paused, state checkpointed)
        → human reviews the proposed action
        → invoke(Command(resume={...}), same thread_id) ▶️
        → approve / reject / edit → agent continues
```

The rest of this notebook demonstrates each of the three decisions in turn.

---

In [10]:
from dotenv import load_dotenv

load_dotenv()

True

In [11]:
from langchain.tools import tool, ToolRuntime


@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]


@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [12]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="gpt-5-nano",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [13]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Please read my email and send a response immediately. Send the reply now in the same thread."
            )
        ],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.",
    },
    config=config,
)

In [14]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem—thanks '
                                                                          'for '
                                                                          'the '
                                                                          'heads-up. '
                                                                          'We '
                                                                          'can '
                                                                          'reschedule. '
                                                                          'Are '
     

In [15]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'Hi John,\n\nNo problem—thanks for the heads-up. We can reschedule. Are you available tomorrow at 10:00 AM or 2:00 PM? If neither works, please suggest a time that suits you, and I’ll adjust. If you’d prefer, we can keep it quick and do a 15-minute catch-up at a time that fits.\n\nBest regards,\nSeán'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'Hi John,\\n\\nNo problem—thanks for the heads-up. We can reschedule. Are you available tomorrow at 10:00 AM or 2:00 PM? If neither works, please suggest a time that suits you, and I’ll adjust. If you’d prefer, we can keep it quick and do a 15-minute catch-up at a time that fits.\\n\\nBest regards,\\nSeán'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='be5a2bfc6c2f21763afc50d43f6cadf2')]


In [16]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John,

No problem—thanks for the heads-up. We can reschedule. Are you available tomorrow at 10:00 AM or 2:00 PM? If neither works, please suggest a time that suits you, and I’ll adjust. If you’d prefer, we can keep it quick and do a 15-minute catch-up at a time that fits.

Best regards,
Seán


## Approve ✅

In [ ]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

## Reject ❌

In [17]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem—thanks '
                                                                          'for '
                                                                          'the '
                                                                          'heads-up. '
                                                                          'We '
                                                                          'can '
                                                                          'reschedule. '
                                                                          'Are '
     

In [18]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John,

No problem—thanks for the heads-up. We can reschedule. Are you available tomorrow at 10:00 AM or 2:00 PM? If neither works, please suggest a time that suits you, and I’ll adjust. If you’d prefer, we can keep it quick and do a 15-minute catch-up at a time that fits.

Your merciful leader, Seán


## Edit

In [ ]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)